In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold,cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,f1_score

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
train=pd.read_csv(r"F:\playground-series-s6e6\train.csv")
test=pd.read_csv(r"F:\playground-series-s6e6\test.csv")

In [3]:
train.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [4]:
test.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [5]:
print(train.shape)
print(test.shape)

(577347, 12)
(247435, 11)


In [6]:
TARGET="class"
ID_COL="id"

train_ids=train[ID_COL]
test_ids=test[ID_COL]

X=train.drop(columns=[TARGET, ID_COL])
y=train[TARGET]
X_test=test.drop(columns=[ID_COL])

In [7]:
target_encoder=LabelEncoder()
y_encoded=target_encoder.fit_transform(y)

In [ ]:
def feature_set_2(df):

    df = df.copy()
    # feature set 1
    df["u_g"]=df["u"] - df["g"]
    df["g_r"]=df["g"] - df["r"]
    df["r_i"]=df["r"] - df["i"]
    df["i_z"]=df["i"] - df["z"]
    df["u_r"]=df["u"] - df["r"]
    df["u_i"]=df["u"] - df["i"]
    df["u_z"]=df["u"] - df["z"]
    df["g_i"]=df["g"] - df["i"]
    df["g_z"]=df["g"] - df["z"]
    df["r_z"]=df["r"] - df["z"]
     
    # Redshift transforms
    df["log_redshift"]=np.log1p(df["redshift"])
    df["sqrt_redshift"]=np.sqrt(np.clip(df["redshift"], 0, None))
    df["redshift_sq"]=(df["redshift"]**2)

    # Redshift × Magnitude
    df["redshift_u"]=(df["redshift"] * df["u"])
    df["redshift_g"]=(df["redshift"] * df["g"])
    df["redshift_r"]=(df["redshift"] * df["r"])
    df["redshift_i"]=(df["redshift"] * df["i"])
    df["redshift_z"]=(df["redshift"] * df["z"])

    # Redshift × Color Indices
    df["redshift_u_g"]=(df["redshift"] * df["u_g"])
    df["redshift_g_r"]=(df["redshift"] * df["g_r"])
    df["redshift_r_i"]=(df["redshift"] * df["r_i"])
    df["redshift_i_z"]=(df["redshift"] * df["i_z"])

    # Ratios
    eps=1e-6
    df["redshift_div_u"]=(df["redshift"] / (np.abs(df["u"]) + eps))
    df["redshift_div_g"]=(df["redshift"] / (np.abs(df["g"]) + eps))
    df["redshift_div_r"]=(df["redshift"] / (np.abs(df["r"]) + eps))
    df["redshift_div_i"]=(df["redshift"] / (np.abs(df["i"]) + eps))
    df["redshift_div_z"]=(df["redshift"] / (np.abs(df["z"]) + eps))
    
    return df

In [61]:
X_fs3=feature_set_3(X)
X_fs3_test=feature_set_3(X_test)

In [62]:
print(X_fs3.shape)
print(X_fs3_test.shape)

(577347, 28)
(247435, 28)


In [ ]:
X_fs3.to_csv("X_fs3.csv", index=False)
X_fs3_test.to_csv("X_test_fs3.csv", index=False)

In [ ]:
cat_cols = ["spectral_type", "galaxy_population"]

ohe = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols
        )
    ],
    remainder="passthrough"
)

X_ohe = ohe.fit_transform(X_fs3)
X_test_ohe = ohe.transform(X_fs3_test)

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
X_mi=pd.get_dummies(X_fs3,drop_first=True)

mi_scores = mutual_info_classif(
    X_mi,
    y_encoded,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": X_mi.columns,
    "MI": mi_scores
})

mi_df.sort_values(
    "MI",
    ascending=False
).head(50)

,Feature,MI
7,redshift,0.514988
16,g_z,0.406561
15,g_i,0.394853
14,u_z,0.382823
13,u_i,0.376866
27,spectral_type_M,0.334415
25,locus_diff,0.325182
9,g_r,0.321154
19,redshift_g_r,0.313814
12,u_r,0.310171


In [ ]:
et = ExtraTreesClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

et.fit(
    X_mi,
    y
)

et_df = pd.DataFrame({
    "Feature": X_mi.columns,
    "Importance": et.feature_importances_
})

et_df.sort_values(
    "Importance",
    ascending=False
).head(50)

,Feature,Importance
7,redshift,0.113468
27,spectral_type_M,0.106109
25,locus_diff,0.102725
29,galaxy_population_Red_Sequence,0.056185
19,redshift_g_r,0.042511
16,g_z,0.039313
14,u_z,0.037537
3,g,0.034823
18,redshift_u_g,0.034584
4,r,0.032955


In [ ]:
lgbm_imp_model = LGBMClassifier(
    random_state=42,
    verbose=-1
)

lgbm_imp_model.fit(
    X_ohe,
    y
)

imp_df = pd.DataFrame({
    "Feature": ohe.get_feature_names_out(),
    "Importance": lgbm_imp_model.feature_importances_
})

imp_df.sort_values(
    "Importance",
    ascending=False
).head(50)

,Feature,Importance
7,remainder__delta,1592
6,remainder__alpha,1574
13,remainder__redshift,1334
12,remainder__z,391
22,remainder__g_z,389
9,remainder__g,388
10,remainder__r,331
8,remainder__u,318
14,remainder__u_g,303
11,remainder__i,281


In [ ]:
lgbm_fs3 = LGBMClassifier(
    subsample=1.0,
    reg_lambda=0.1,
    reg_alpha=1,
    num_leaves=127,
    n_estimators=1000,
    min_child_samples=10,
    max_depth=-1,
    learning_rate=0.03,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

lgbm_scores_fs3 = cross_val_score(
    lgbm_fs3,
    X_ohe,
    y_encoded,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("LightGBM FS2")
print("Mean:", lgbm_scores_fs3.mean())
print("Std :", lgbm_scores_fs3.std())

LightGBM FS2
Mean: 0.9681993667444753
Std : 0.000241978800762592


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

mlp = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("model", MLPClassifier(
        hidden_layer_sizes=(512,256,128),
        learning_rate_init=0.001,
        max_iter=300,
        random_state=42
    ))
])

scores = cross_val_score(
    mlp,
    X_ohe,
    y_encoded,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print(scores.mean())

KeyboardInterrupt: 